In [1]:
!pip install pandas numpy scikit-learn nltk streamlit matplotlib seaborn wordcloud joblib

In [2]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
fake = pd.read_csv("../data/Fake.csv") # Fake haber datasetini okur.
true = pd.read_csv("../data/True.csv") # Gerçek haber datasetini okur.

In [4]:
fake["label"] = 0  # Fake haberlere etiket verir.
true["label"] = 1  # Gerçek haberlere etiket verir.
  

In [5]:
df = pd.concat([fake, true], axis=0) # İki dataframe’i alt alta birleştirir.

In [6]:
df = df.sample(frac=1, random_state=42) # Verileri karıştırır.

In [7]:
df.reset_index(drop=True, inplace=True) # Karıştırmadan sonra indexleri sıfırlar.

df.info() Şunları gösterir:

* sütun isimleri
* veri tipleri
* boş veri var mı

In [8]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   label    44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 112.3 MB


In [9]:
df.isnull().sum() # Her sütunda kaç boş veri olduğunu gösterir.

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [10]:
df.dropna(inplace=True) # Boş satırları siler.

In [11]:
df.duplicated().sum() # Kaç tane tekrar eden veri var gösterir.

np.int64(209)

In [12]:
df.drop_duplicates(inplace=True) # Tekrarlayan haberleri kaldırır.

In [13]:
df.shape # Kaç satır ve sütun olduğunu gösterir.

(44689, 5)

In [14]:
def clean_text(text): # Metni temizleyen fonksiyon.

    text = str(text).lower() # Bütün harfleri küçültür.

    # url kaldır
    text = re.sub(r"http\S+", "", text) # Linkleri kaldırır.

    # www kaldır
    text = re.sub(r"www\S+", "", text)

    # html tag kaldır
    text = re.sub(r"<.*?>", "", text)

    # noktalama kaldır
    text = re.sub(r"[^\w\s]", "", text)

    # fazla boşluk kaldır
    text = re.sub(r"\s+", " ", text).strip() # Birden fazla boşluğu temizler.

    return text

In [15]:
df["clean_text"] = df["text"].apply(clean_text) # Temizlenmiş metni yeni sütuna yazar.

In [16]:
df[["text", "clean_text"]].head() # Orijinal ve temizlenmiş metni karşılaştırır.

,text,clean_text
0,"21st Century Wire says Ben Stein, reputable pr...",21st century wire says ben stein reputable pro...
1,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters us president donald trump r...
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...
3,"On Monday, Donald Trump once again embarrassed...",on monday donald trump once again embarrassed ...
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",glasgow scotland reuters most us presidential ...


In [17]:
import nltk



In [18]:
nltk.download("stopwords") # İngilizce anlamsız kelimeleri indirir.

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
from nltk.corpus import stopwords

In [20]:
stop_words = set(stopwords.words("english")) # İngilizce gereksiz kelimeleri alır.

In [21]:
def remove_stopwords(text): # Anlamsız kelimeleri kaldırır.

    words = text.split() # Cümleyi kelimelere böler.

    filtered_words = [  # Stopword olmayanları bırakır.
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words) # Kelimeleri tekrar cümle yapar.

In [22]:
df["clean_text"] = df["clean_text"].apply(remove_stopwords) # Temizleme işlemini tüm veriye uygular.

In [23]:
df[["text", "clean_text"]].head()

,text,clean_text
0,"21st Century Wire says Ben Stein, reputable pr...",21st century wire says ben stein reputable pro...
1,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters us president donald trump r...
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...
3,"On Monday, Donald Trump once again embarrassed...",monday donald trump embarrassed country accide...
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",glasgow scotland reuters us presidential candi...


In [24]:
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [25]:
from nltk.stem import WordNetLemmatizer


In [26]:

lemmatizer = WordNetLemmatizer() # Kelime kökü bulma aracı.

In [27]:
def lemmatize_text(text): # Kelimeleri kök haline çevirir.

    words = text.split()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(lemmatized_words)

In [28]:
df["clean_text"] = df["clean_text"].apply(lemmatize_text) # Tüm dataset’e uygular.

In [29]:
df[["text", "clean_text"]].head()

,text,clean_text
0,"21st Century Wire says Ben Stein, reputable pr...",21st century wire say ben stein reputable prof...
1,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters u president donald trump re...
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...
3,"On Monday, Donald Trump once again embarrassed...",monday donald trump embarrassed country accide...
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",glasgow scotland reuters u presidential candid...


In [30]:
df["text_length"] = df["clean_text"].apply(lambda x: len(x.split())) # Her haberin kaç kelime olduğunu hesaplar.

df["text_length"].describe() Şunları verir:

* ortalama
* min
* max
* standart sapma

In [31]:
df["text_length"].describe()

count    44689.000000
mean       233.022981
std        203.499615
min          0.000000
25%        118.000000
50%        205.000000
75%        292.000000
max       4920.000000
Name: text_length, dtype: float64

In [32]:
df = df[df["text_length"] > 5] # 5 kelimeden kısa haberleri kaldırır.

In [33]:
df.shape # Temizlik sonrası veri boyutunu gösterir.

(43822, 7)

In [34]:
df.to_csv("../data/cleaned_news.csv", index=False) # Temizlenmiş veriyi CSV olarak kaydeder.